# Titanic · ансамбли

Проверяем, улучшает ли объединение выбранных RF, XGBoost и Logistic Regression качество отдельных моделей. Основная метрика — Accuracy, дополнительная — ROC-AUC. Порог 0.5, веса усреднения равные; нового подбора нет.

**Маршрут:** модели и общие фолды → OOF → усреднение → hard voting → stacking → проверка других разбиений. Подготовка не обучает модели; последующие разделы явно запускают обучение.

**Запуск:** сверху вниз. Таблица сравнения при seed 42 — после stacking, итоговая проверка seeds 42/17/123 — в конце. Проверка отдельных моделей находится в [model_stability.ipynb](model_stability.ipynb).


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config.yaml").is_file() or not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Откройте ноутбук из корня Titanic-ML или папки notebooks")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier

from src.sklearn_experiments import build_experiment, get_cv
from sklearn.base import clone
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold


df_train = pd.read_csv("data/train.csv")


## Выбранные модели и обработка

Для обеих моделей используем семь исходных признаков, числовую медиану и one-hot для категорий, без масштабирования. `Sex` кодируется с удалением одного бинарного столбца; `Pclass` остаётся числовым. Это обработка из выбранных исследований, а не новая конфигурация.


In [2]:
rf_config = {
    "features": {
        "Age": "numeric",
        "Fare": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": False,
}

best_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features=None,
    min_samples_leaf=4,
    min_samples_split=10,
    random_state=42,
)


In [3]:
xgb_config = {
    "features": rf_config["features"].copy(),
    "scale_numeric": False,
}

best_xgb = XGBClassifier(
    n_estimators=80,
    learning_rate=0.1,
    max_depth=4,
    min_child_weight=5,
    random_state=42,
    eval_metric="logloss",
    n_jobs=1,
)

X, y, rf_pipeline = build_experiment(
    df=df_train,
    target="Survived",
    model=best_rf,
    config=rf_config,
)
_, _, xgb_pipeline = build_experiment(
    df=df_train,
    target="Survived",
    model=best_xgb,
    config=xgb_config,
)

model_pipelines = {
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
}


## Общие фолды

Один список `cv_splits` используем для обеих моделей: пять стратифицированных разбиений с перемешиванием и seed 42. Индексы в списке — позиции строк: при обходе выбираем данные через `.iloc`.

На каждом фолде понадобится свежая копия pipeline (`clone`), обучение на `train_indices` и вероятности класса 1 через `predict_proba` на `validation_indices`. Здесь только готовим разбиения.


In [4]:
cv_splits = list(get_cv().split(X, y))
threshold = 0.5

fold_summary = pd.DataFrame([
    {
        "fold": fold,
        "train_samples": len(train_indices),
        "validation_samples": len(validation_indices),
        "validation_survival_rate": y.iloc[validation_indices].mean(),
    }
    for fold, (train_indices, validation_indices) in enumerate(cv_splits, start=1)
])
fold_summary


,fold,train_samples,validation_samples,validation_survival_rate
0,1,712,179,0.385475
1,2,713,178,0.382022
2,3,713,178,0.382022
3,4,713,178,0.382022
4,5,713,178,0.387640


## OOF-предсказания и оценка усреднения

**OOF (out-of-fold)** — предсказания для строк, которые не участвовали в обучении предсказывающей модели. На каждом фолде заполняем вероятности только для его проверочных строк на их исходных позициях. После пяти фолдов у каждого пассажира будет одна проверочная вероятность от RF и одна от XGBoost.

Из этих вероятностей получим классы по порогу 0.5, сравним общие ошибки и проверим усреднение с равными весами. OOF исключает обучение модели на предсказываемой строке, но эта оценка не является независимым финальным тестом: настройки моделей ранее выбирались на этих данных.


In [5]:
pair_pipelines = {name: model_pipelines[name] for name in ["Random Forest", "XGBoost"]}
predictions = {name: np.full(len(X), np.nan) for name in pair_pipelines}

for fold, (train_indices, validation_indices) in enumerate(cv_splits, start=1):
    train_X = X.iloc[train_indices]
    train_y = y.iloc[train_indices]
    validation_X = X.iloc[validation_indices]

    for name, pipeline in pair_pipelines.items():
        model = clone(pipeline)
        model.fit(train_X, train_y)
        probabilities = model.predict_proba(validation_X)[:, 1]
        predictions[name][validation_indices] = probabilities


for name, model_probabilities in predictions.items():
    assert len(model_probabilities) == len(X), f"{name}: неверная длина"
    assert np.isfinite(model_probabilities).all(), f"{name}: остались NaN или бесконечные значения"
    assert ((model_probabilities >= 0) & (model_probabilities <= 1)).all(), (
        f"{name}: вероятности вне диапазона [0, 1]"
    )

## Добавление Logistic Regression

Используем выбранные в `modeling.ipynb` признаки и параметры: C=1, L2, solver=saga. У LR свой набор признаков и масштабирование внутри pipeline. Порядок пассажиров и проверочные фолды совпадают с RF и XGBoost.

Следующие две ячейки обучают только LR: уже собранные `predictions` для RF и XGBoost сохраняются. После их выполнения запускаем общую оценку. Для полного запуска ноутбука выполняем ячейки сверху вниз; повторно запускать ячейку обучения пары после добавления LR не требуется.


In [6]:
logreg_best_config = {
    "features": {
        "Age": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "onehot",
        "_Fare": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },

    "fare_log1p": True,
    "sex_pclass": True,
    "age_squared": True,
    "is_alone": True,

    "is_child": False,
    "family_size": False,
    "scale_numeric": True,
}

best_logreg = LogisticRegression(
    solver="saga",
    C=1,
    l1_ratio=0.0,
    max_iter=1000,
    random_state=42,
)

X_logreg, y_logreg, logreg_pipeline = build_experiment(
    df=df_train,
    target="Survived",
    model=best_logreg,
    config=logreg_best_config,
)
assert X_logreg.index.equals(X.index)
assert y_logreg.equals(y)

model_pipelines["Logistic Regression"] = logreg_pipeline


In [7]:
logreg_probabilities = np.full(len(X), np.nan)
for fold, (train_indices, validation_indices) in enumerate(cv_splits, start=1):
    model = clone(model_pipelines["Logistic Regression"])
    model.fit(X_logreg.iloc[train_indices], y.iloc[train_indices])
    logreg_probabilities[validation_indices] = model.predict_proba(
        X_logreg.iloc[validation_indices]
    )[:, 1]

assert np.isfinite(logreg_probabilities).all(), "LR: остались NaN или бесконечные значения"
assert ((logreg_probabilities >= 0) & (logreg_probabilities <= 1)).all()
predictions["Logistic Regression"] = logreg_probabilities


## Усреднение вероятностей

Считаем вероятности RF + XGBoost и RF + XGBoost + LR с равными весами. Accuracy использует классы по порогу 0.5, ROC-AUC — вероятности. Оценки по пяти фолдам сохраняются в fold_results; общая таблица появится после stacking.


In [8]:
evaluation_probabilities = {
    **predictions,
    "RF + XGBoost": (
        predictions["Random Forest"] + predictions["XGBoost"]
    ) / 2,
    "RF + XGBoost + LR": (
        predictions["Random Forest"]
        + predictions["XGBoost"]
        + predictions["Logistic Regression"]
    ) / 3,
}

oof_predictions = pd.DataFrame(evaluation_probabilities, index=X.index)
oof_predictions.insert(0, "Survived", y)

fold_results = []
for fold, (_, validation_indices) in enumerate(cv_splits, start=1):
    validation_y = y.iloc[validation_indices]
    for name, model_probabilities in evaluation_probabilities.items():
        probabilities = model_probabilities[validation_indices]
        predicted_classes = (probabilities >= threshold).astype(int)
        fold_results.append({
            "model": name,
            "fold": fold,
            "accuracy": accuracy_score(validation_y, predicted_classes),
            "roc_auc": roc_auc_score(validation_y, probabilities),
        })

fold_results = pd.DataFrame(fold_results)
ensemble_summary = fold_results.groupby("model", sort=False).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", lambda values: values.std(ddof=0)),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", lambda values: values.std(ddof=0)),
)



## Hard voting: большинство голосов

Каждая из трёх моделей голосует классом по порогу 0.5. Итог — класс 1, если за него проголосовали хотя бы две модели. Используем готовые OOF-вероятности; новых обучений в этом блоке нет.

Сравниваем Accuracy на прежних фолдах. ROC-AUC здесь не считаем: результат голосования — класс, а не вероятность.


In [9]:
voting_models = ["Random Forest", "XGBoost", "Logistic Regression"]
model_votes = np.column_stack([
    predictions[name] >= threshold for name in voting_models
])
hard_voting_classes = (model_votes.sum(axis=1) >= 2).astype(int)

hard_voting_results = pd.DataFrame([
    {
        "model": "Hard voting (RF + XGBoost + LR)",
        "fold": fold,
        "accuracy": accuracy_score(
            y.iloc[validation_indices], hard_voting_classes[validation_indices]
        ),
    }
    for fold, (_, validation_indices) in enumerate(cv_splits, start=1)
])

voting_fold_results = pd.concat([
    fold_results[["model", "fold", "accuracy"]],
    hard_voting_results,
], ignore_index=True)
voting_summary = voting_fold_results.groupby("model", sort=False).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", lambda values: values.std(ddof=0)),
)


На разбиении seed 42 голосование и усреднение не превзошли RF. Далее проверяем обучаемое объединение — stacking; сравнение всех вариантов приводим одной таблицей.


## Stacking: обучение объединяющей модели

Базовые модели — выбранные RF, XGBoost и LR. Метамодель — отдельная Logistic Regression с C=1 и L2, без подбора. Она получает только три вероятности (`passthrough=False`).

На каждом внешнем train-фолде `StackingClassifier` создаёт внутренние OOF на пяти стратифицированных фолдах и обучает на них метамодель. Затем базовые модели, обученные на всём внешнем train, дают вероятности для внешнего validation; метамодель объединяет их. Внешний validation не участвует ни в обучении обработки, ни в обучении моделей.

Общий вход содержит объединение колонок RF/XGBoost и LR. Производные признаки вычислены построчно; медианы, кодировщики и scaler обучаются внутри каждого pipeline. Оценка использует прежние внешние `cv_splits`. Сохранённые глобальные OOF не используем для обучения метамодели: внутренние OOF создаются заново внутри внешнего train.


In [10]:
extra_logreg_features = X_logreg.columns.difference(X.columns, sort=False)
X_stacking = X.join(X_logreg[extra_logreg_features])
assert X_stacking.index.equals(y.index)

stacking_model = StackingClassifier(
    estimators=[
        ("rf", clone(rf_pipeline)),
        ("xgb", clone(xgb_pipeline)),
        ("lr", clone(logreg_pipeline)),
    ],
    final_estimator=LogisticRegression(C=1, max_iter=1000, random_state=42),
    cv=get_cv(),
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=1,
)

stacking_probabilities = np.full(len(X), np.nan)
stacking_fold_results = []
for fold, (train_indices, validation_indices) in enumerate(cv_splits, start=1):
    model = clone(stacking_model)
    model.fit(X_stacking.iloc[train_indices], y.iloc[train_indices])
    probabilities = model.predict_proba(X_stacking.iloc[validation_indices])[:, 1]
    stacking_probabilities[validation_indices] = probabilities
    stacking_fold_results.append({
        "model": "Stacking (RF + XGBoost + LR)",
        "fold": fold,
        "accuracy": accuracy_score(y.iloc[validation_indices], probabilities >= threshold),
        "roc_auc": roc_auc_score(y.iloc[validation_indices], probabilities),
    })
    print(f"Stacking: внешний фолд {fold}/{len(cv_splits)} завершён")

assert np.isfinite(stacking_probabilities).all()
assert ((stacking_probabilities >= 0) & (stacking_probabilities <= 1)).all()
stacking_fold_results = pd.DataFrame(stacking_fold_results)
oof_predictions["Stacking"] = stacking_probabilities

all_ensemble_fold_results = pd.concat([
    fold_results,
    hard_voting_results,
    stacking_fold_results,
], ignore_index=True)
all_ensemble_summary = all_ensemble_fold_results.groupby("model", sort=False).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", lambda values: values.std(ddof=0)),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", lambda values: values.std(ddof=0)),
)
all_ensemble_summary


Stacking: внешний фолд 1/5 завершён
Stacking: внешний фолд 2/5 завершён
Stacking: внешний фолд 3/5 завершён
Stacking: внешний фолд 4/5 завершён
Stacking: внешний фолд 5/5 завершён


,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std
model,,,,
Random Forest,0.846212,0.021072,0.881090,0.025175
XGBoost,0.842866,0.020767,0.875820,0.026474
Logistic Regression,0.815944,0.008092,0.863932,0.012152
RF + XGBoost,0.843990,0.025506,0.880945,0.025203
RF + XGBoost + LR,0.840619,0.021265,0.881533,0.022447
Hard voting (RF + XGBoost + LR),0.845088,0.021080,NaN,NaN
Stacking (RF + XGBoost + LR),0.838372,0.021519,0.880888,0.023742


## Сравнение при seed разбиения 42

Stacking получил Accuracy 83.84%, ROC-AUC 0.88089; RF — 84.62%. В этой проверке ансамбли RF не улучшили. Для hard voting ROC-AUC отсутствует: голосование возвращает классы.

Это промежуточный вывод для одного набора фолдов. Ниже проверяем зависимость от разбиения; stacking повторно не обучаем.


## Устойчивость к разбиению: seeds 42, 17 и 123

Меняем только seed пяти стратифицированных фолдов; seed моделей, параметры, обработка и порог остаются прежними. Seed 42 переиспользуем, для 17 и 123 выполняем 30 дополнительных обучений. Stacking не повторяем.

В итоговой таблице колонки 42/17/123 — средние Accuracy по пяти фолдам; accuracy_mean — среднее трёх значений, accuracy_split_std — std между разбиениями (ddof=0). Три разбиения используют одних пассажиров и не являются независимыми тестами.


In [11]:
seed_fold_results = pd.concat([
    fold_results,
    hard_voting_results,
], ignore_index=True).assign(split_seed=42)

seed_model_inputs = {
    "Random Forest": X,
    "XGBoost": X,
    "Logistic Regression": X_logreg,
}
seed_oof_predictions = {42: oof_predictions[voting_models].copy()}

for split_seed in [17, 123]:
    seed_splits = list(StratifiedKFold(
        n_splits=5, shuffle=True, random_state=split_seed
    ).split(X, y))
    seed_predictions = {
        name: np.full(len(X), np.nan) for name in voting_models
    }

    for train_indices, validation_indices in seed_splits:
        for name in voting_models:
            model_X = seed_model_inputs[name]
            model = clone(model_pipelines[name])
            model.fit(model_X.iloc[train_indices], y.iloc[train_indices])
            seed_predictions[name][validation_indices] = model.predict_proba(
                model_X.iloc[validation_indices]
            )[:, 1]

    for name, probabilities in seed_predictions.items():
        assert np.isfinite(probabilities).all(), f"{split_seed}, {name}: неконечные вероятности"
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
    seed_oof_predictions[split_seed] = pd.DataFrame(seed_predictions, index=X.index)

    seed_probabilities = {
        **seed_predictions,
        "RF + XGBoost": (
            seed_predictions["Random Forest"] + seed_predictions["XGBoost"]
        ) / 2,
        "RF + XGBoost + LR": np.mean(np.stack([
            seed_predictions[name] for name in voting_models
        ]), axis=0),
    }
    votes = np.column_stack([
        seed_predictions[name] >= threshold for name in voting_models
    ])
    voting_classes = (votes.sum(axis=1) >= 2).astype(int)

    seed_rows = []
    for fold, (_, validation_indices) in enumerate(seed_splits, start=1):
        labels = y.iloc[validation_indices]
        for name, probabilities in seed_probabilities.items():
            fold_probabilities = probabilities[validation_indices]
            seed_rows.append({
                "split_seed": split_seed,
                "model": name,
                "fold": fold,
                "accuracy": accuracy_score(labels, fold_probabilities >= threshold),
                "roc_auc": roc_auc_score(labels, fold_probabilities),
            })
        seed_rows.append({
            "split_seed": split_seed,
            "model": "Hard voting (RF + XGBoost + LR)",
            "fold": fold,
            "accuracy": accuracy_score(labels, voting_classes[validation_indices]),
            "roc_auc": np.nan,
        })

    seed_fold_results = pd.concat([
        seed_fold_results, pd.DataFrame(seed_rows)
    ], ignore_index=True)
    print(f"Seed разбиения {split_seed}: 15 обучений завершены")

seed_results = seed_fold_results.groupby(
    ["model", "split_seed"], sort=False
).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_fold_std=("accuracy", lambda values: values.std(ddof=0)),
    roc_auc_mean=("roc_auc", "mean"),
).reset_index()
seed_accuracy_comparison = seed_results.pivot(
    index="model", columns="split_seed", values="accuracy_mean"
).reindex(voting_summary.index)[[42, 17, 123]]


Seed разбиения 17: 15 обучений завершены
Seed разбиения 123: 15 обучений завершены


In [12]:
seed_stability_summary = seed_results.groupby("model", sort=False).agg(
    accuracy_mean=("accuracy_mean", "mean"),
    accuracy_split_std=("accuracy_mean", lambda values: values.std(ddof=0)),
    roc_auc_mean=("roc_auc_mean", "mean"),
)
seed_comparison_report = seed_accuracy_comparison.join(seed_stability_summary)
seed_comparison_report.round(5)


,42,17,123,accuracy_mean,accuracy_split_std,roc_auc_mean
model,,,,,,
Random Forest,0.84621,0.82601,0.83393,0.83538,0.00831,0.87214
XGBoost,0.84287,0.83274,0.83280,0.83613,0.00476,0.87095
Logistic Regression,0.81594,0.81592,0.81148,0.81445,0.00210,0.86193
RF + XGBoost,0.84399,0.82264,0.82945,0.83203,0.00890,0.87403
RF + XGBoost + LR,0.84062,0.82714,0.82832,0.83203,0.00610,0.87691
Hard voting (RF + XGBoost + LR),0.84509,0.83162,0.83168,0.83613,0.00634,NaN


## Итог и ограничения

По трём разбиениям RF получил **83.54%**, XGBoost и hard voting — **83.61%**. Разница около 0.08 п.п. не даёт основания объявлять уверенного победителя. Обе смеси вероятностей дали около 83.20%: устойчивого прироста Accuracy не получили.

Stacking проверен только при seed разбиения 42 и RF не превзошёл. Все три учебных подхода — усреднение (soft voting), hard voting и stacking — выполнены. RF остаётся практическим кандидатом, но его превосходство над XGBoost не доказано.

Исследовали изменение разбиения, не seed обучения. Конфигурации ранее выбирались на этих данных; повторные фолды не создают независимый тест. Нового подбора комбинаций, весов и порогов не проводим.

### Финальная проверка на Kaggle

| Сабмит | Public Accuracy |
|---|---:|
| Один RF на всём train | **0.77751** |
| Среднее вероятностей пяти RF, порог 0.5 | 0.76794 |
| Один XGBoost / среднее пяти XGBoost | 0.77751 / 0.77751 |
| Одна EmbeddingMLP / среднее пяти EmbeddingMLP | 0.77751 / 0.77751 |

У двух RF-сабмитов параметры и обработка одинаковые. Усреднение изменило четыре класса и снизило публичную оценку на 0.96 п.п. Итоговым оставлен сабмит одной модели. Это отдельная проверка на Kaggle; CV-метрики выше не являются оценкой ансамбля из пяти RF на тесте.

Новые сабмиты не превзошли одиночный RF; итоговый выбор сохраняется. У XGBoost усреднение изменило 6 ответов, у DNN — 15, но Accuracy осталась прежней. Повторение: [scripts/additional_submissions.py](../scripts/additional_submissions.py); условия и результаты — в [журнале](../RESEARCH_SUMMARY.md).
